# SCOPE — fine-tuning the Vision Transformer

Trains the evidence classifier used at **S10**: given a photograph, return
P(smoking violation).

**Run this in Google Colab with a GPU runtime.** Runtime → Change runtime type →
T4 GPU. On CPU a single epoch takes hours; on a T4 it takes minutes.

### Why KerasHub and not HuggingFace `transformers`

`transformers` **removed TensorFlow support** — `TFViTForImageClassification`
no longer exists. KerasHub is the Keras-native replacement, runs on the
TensorFlow backend, and keeps the project honest to Table 3.5 of the report
(*TensorFlow / Keras*). Nothing here needs PyTorch.

### Before you start

Run `python ml/prepare_dataset.py` locally first. This notebook consumes the
`manifest.csv` it produces — the split is decided there, deliberately, so that
it is reproducible and free of train/test leakage rather than re-rolled here.


## 1. Environment


In [ ]:
!pip install -q keras-hub keras --upgrade

import os
os.environ['KERAS_BACKEND'] = 'tensorflow'   # Table 3.5 commits to TensorFlow

import json, pathlib
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_hub

print('keras     ', keras.__version__)
print('tensorflow', tf.__version__)
print('GPU       ', tf.config.list_physical_devices('GPU') or 'NONE — switch runtime to T4 GPU')


## 2. Data

Upload a zip containing `manifest.csv` and the `data/raw/` tree, or mount Drive.
The manifest's `path` column is relative to the `ml/` directory.


In [ ]:
from google.colab import files

# Option A — upload a zip of the ml/data directory
uploaded = files.upload()          # choose e.g. scope-data.zip
!unzip -q -o $(ls *.zip | head -1) -d /content/ml

# Option B — Google Drive instead (comment out Option A above)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r '/content/drive/MyDrive/scope/ml' /content/ml

ML_DIR = pathlib.Path('/content/ml')
manifest = pd.read_csv(ML_DIR / 'data' / 'manifest.csv')
print(manifest.groupby(['split', 'label']).size())


### Verify the split before training

Cheap, and it protects the headline result: if a perceptual hash appeared in
two splits, the reported test accuracy would be inflated by memorisation.


In [ ]:
spanning = manifest.groupby('phash')['split'].nunique()
assert (spanning <= 1).all(), 'LEAKAGE: a hash group spans multiple splits'
assert manifest['path'].is_unique, 'duplicate paths in manifest'
print('No leakage between splits.')
print('Total images:', len(manifest))


## 3. Input pipeline

Augmentation is applied to the **training split only** — augmenting validation
or test would measure the model against images it will never see in the field.

There is no vertical flip: a cigarette has an orientation, and an upside-down
photograph is not a case this app will ever encounter.


In [ ]:
IMAGE_SIZE = 224          # ViT base patch16 expects 224x224
BATCH_SIZE = 32
CLASS_NAMES = ['notsmoking', 'smoking']   # index 1 == positive class
AUTOTUNE = tf.data.AUTOTUNE


def decode(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
    return tf.cast(img, tf.float32), label


def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    return tf.clip_by_value(img, 0.0, 255.0), label


def build(split, training):
    rows = manifest[manifest['split'] == split]
    paths = [str(ML_DIR / p) for p in rows['path']]
    labels = [CLASS_NAMES.index(l) for l in rows['label']]

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=20260813, reshuffle_each_iteration=True)
    ds = ds.map(decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = build('train', training=True)
val_ds   = build('val',   training=False)
test_ds  = build('test',  training=False)

images, labels = next(iter(train_ds))
print('batch', images.shape, 'range', float(tf.reduce_min(images)), '-', float(tf.reduce_max(images)))


## 4. Model

A ViT pre-trained on ImageNet-21k, with a fresh 2-class head. The dataset here
is far too small to train a transformer from scratch — transfer learning is what
makes this feasible at all, and is the approach Mazurets et al. (2022) used for
the same task.


In [ ]:
PRESET = 'vit_base_patch16_224_imagenet21k'

classifier = keras_hub.models.ViTImageClassifier.from_preset(
    PRESET,
    num_classes=2,
)

# Stage 1: freeze the backbone. The new head starts random, and letting large
# random gradients flow into pre-trained weights destroys the features that
# make transfer learning work.
classifier.backbone.trainable = False

classifier.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)
classifier.summary()


## 5. Stage 1 — train the head


In [ ]:
EPOCHS_HEAD = 5

history_head = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True),
    ],
)


## 6. Stage 2 — fine-tune

Unfreeze the backbone at a much lower learning rate. 1e-3 would wreck the
pre-trained weights; 1e-5 adapts them.


In [ ]:
classifier.backbone.trainable = True

classifier.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

EPOCHS_FINE = 10
history_fine = classifier.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2),
    ],
)


## 7. Training curves

Chapter 5 needs these. A validation loss that rises while training loss keeps
falling is overfitting — collect more data or stop earlier.


In [ ]:
import matplotlib.pyplot as plt

def series(key):
    return list(history_head.history[key]) + list(history_fine.history[key])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (a, b, title) in zip(axes, [('accuracy','val_accuracy','Accuracy'),
                                    ('loss','val_loss','Loss')]):
    ax.plot(series(a), label='train')
    ax.plot(series(b), label='validation')
    ax.axvline(len(history_head.history[a]) - 0.5, ls='--', c='grey', lw=1)
    ax.set_title(title + '  (dashed = backbone unfrozen)')
    ax.set_xlabel('epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()


## 8. Evaluate on the held-out test split

**This split has not influenced training or early stopping.** It is touched
once, here. Re-tuning anything after looking at these numbers and then
re-reporting them would make the figure meaningless.

The per-image scores written below are the input to `ml/select_threshold.py`,
which chooses the operating point.


In [ ]:
loss, accuracy = classifier.evaluate(test_ds)
print(f'Test loss {loss:.4f}   accuracy {accuracy:.4f}')

probs = classifier.predict(test_ds)
positive = probs[:, CLASS_NAMES.index('smoking')]

test_rows = manifest[manifest['split'] == 'test'].reset_index(drop=True)
scores = pd.DataFrame({'label': test_rows['label'], 'score': positive})
scores.to_csv('/content/test_scores.csv', index=False)

print(scores.groupby('label')['score'].describe()[['count', 'mean', 'min', 'max']])
files.download('/content/test_scores.csv')
files.download('/content/training_curves.png')


## 9. Export the model

Saved in Keras 3 format and loaded by `backend/app/services/inference.py`.
Download it, unzip into `ml/artifacts/`, and set `AI_MODEL_PATH` in
`backend/.env`.

Record `MODEL_VERSION` in `backend/.env` too — `AI_CLASSIFICATION.model_version`
stores it per report, so a later retrain never silently changes the meaning of
a historical confidence score.


In [ ]:
MODEL_VERSION = 'vit-scope-v1'

classifier.save(f'/content/{MODEL_VERSION}.keras')

metadata = {
    'model_version': MODEL_VERSION,
    'preset': PRESET,
    'image_size': IMAGE_SIZE,
    'class_names': CLASS_NAMES,
    'positive_index': CLASS_NAMES.index('smoking'),
    'test_accuracy': float(accuracy),
    'train_images': int((manifest['split'] == 'train').sum()),
    'val_images': int((manifest['split'] == 'val').sum()),
    'test_images': int((manifest['split'] == 'test').sum()),
}
pathlib.Path('/content/model_metadata.json').write_text(json.dumps(metadata, indent=2))
print(json.dumps(metadata, indent=2))

files.download(f'/content/{MODEL_VERSION}.keras')
files.download('/content/model_metadata.json')


## 10. Next

1. Put `test_scores.csv` in `ml/artifacts/`
2. Run `python ml/select_threshold.py` — it recommends the threshold and prints
   the table for Chapter 5
3. Set `AI_CONFIDENCE_THRESHOLD` and `AI_MODEL_VERSION` in `backend/.env`
4. Replace the stub in `backend/app/services/inference.py` with real inference

Keep `training_curves.png`, the composition table from `ml/data/report.md`, and
the threshold sweep — together they are the evaluation evidence for Chapter 5.
